# 09 — Secondary Safety Evaluation Summary Across Horizons

This notebook collects the outputs produced by `07_Secondary_Safety_Evaluation.ipynb` for all prediction horizons and creates summary tables and comparison plots.

Run `07_Secondary_Safety_Evaluation.ipynb` once for each horizon first:

```python
HORIZON = 1
HORIZON = 15
HORIZON = 25
HORIZON = 50
HORIZON = 75
HORIZON = 125
```

Then run this notebook. It will automatically find the saved secondary-evaluation CSVs.

This version uses multi-panel figures: two or four plots per image, with one shared legend outside the plotting area.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Imports ready.")

In [ ]:
# ============================================================
# CONFIG
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"
SECONDARY_DIR = DATA_DIR / "secondary_safety_evaluation"
SUMMARY_DIR = DATA_DIR / "secondary_safety_all_horizons_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

FPS = 25
HISTORY = 25
STRIDE = 1

# You used 15 frames instead of 13.
HORIZONS = [1, 15, 25, 50, 75, 125]

THRESHOLD_TAG = "brake0p1_accel0p2"
IDM_PARAM_TAG = "refinedIDM_v2"

assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"
assert SECONDARY_DIR.exists(), f"SECONDARY_DIR does not exist: {SECONDARY_DIR}"

print("Data dir:", DATA_DIR)
print("Secondary dir:", SECONDARY_DIR)
print("Summary dir:", SUMMARY_DIR)
print("Horizons:", HORIZONS)

In [ ]:
# ============================================================
# MODEL LABELS / GROUPS
# ============================================================

MODEL_INFO = {
    # IDM baselines
    "Global IDM": {
        "Family": "IDM",
        "Type": "Deployable",
        "ShortName": "Global IDM",
    },
    "XGB-soft best-IDM selector": {
        "Family": "IDM",
        "Type": "Deployable",
        "ShortName": "XGB-soft IDM",
    },
    "True-regime IDM": {
        "Family": "IDM",
        "Type": "Upper-bound",
        "ShortName": "True-regime IDM",
    },
    "Ideal best-candidate IDM": {
        "Family": "IDM",
        "Type": "Upper-bound",
        "ShortName": "Ideal IDM",
    },

    # Plain Transformer
    "Plain Transformer": {
        "Family": "Transformer",
        "Type": "Deployable",
        "ShortName": "Transformer",
    },

    # IDM-first residual hybrids
    "Residual Hybrid + Global IDM": {
        "Family": "IDM-first residual hybrid",
        "Type": "Deployable",
        "ShortName": "Residual + Global IDM",
    },
    "Residual Hybrid + Tree-selected IDM": {
        "Family": "IDM-first residual hybrid",
        "Type": "Deployable",
        "ShortName": "Residual + Tree IDM",
    },
    "Residual Hybrid + XGB-soft best-IDM": {
        "Family": "IDM-first residual hybrid",
        "Type": "Deployable",
        "ShortName": "Residual + XGB IDM",
    },
    "Residual Hybrid + True-regime IDM": {
        "Family": "IDM-first residual hybrid",
        "Type": "Upper-bound",
        "ShortName": "Residual + True-regime IDM",
    },

    # Transformer weighted fusion
    "Transformer Weighted Fusion + Global IDM": {
        "Family": "Transformer weighted fusion",
        "Type": "Deployable",
        "ShortName": "Fusion + Global IDM",
    },
    "Transformer Weighted Fusion + Tree-selected IDM": {
        "Family": "Transformer weighted fusion",
        "Type": "Deployable",
        "ShortName": "Fusion + Tree IDM",
    },
    "Transformer Weighted Fusion + True-regime IDM": {
        "Family": "Transformer weighted fusion",
        "Type": "Upper-bound",
        "ShortName": "Fusion + True-regime IDM",
    },

    # Transformer with IDM feature
    "Transformer + XGB-soft best-IDM feature": {
        "Family": "Transformer + IDM feature",
        "Type": "Deployable",
        "ShortName": "Transformer + XGB-IDM feature",
    },
}


def add_model_info(df):
    out = df.copy()

    out["Family"] = out["Model"].map(
        lambda m: MODEL_INFO.get(m, {}).get("Family", "Unknown")
    )
    out["Type"] = out["Model"].map(
        lambda m: MODEL_INFO.get(m, {}).get("Type", "Unknown")
    )
    out["ShortName"] = out["Model"].map(
        lambda m: MODEL_INFO.get(m, {}).get("ShortName", m)
    )

    return out


DEFAULT_MODELS_FOR_PLOTS = [
    "Plain Transformer",
    "Global IDM",
    "XGB-soft best-IDM selector",
    "Residual Hybrid + Global IDM",
    "Residual Hybrid + XGB-soft best-IDM",
    "Transformer Weighted Fusion + Global IDM",
    "Transformer + XGB-soft best-IDM feature",
    "Residual Hybrid + True-regime IDM",
]

print("Model metadata ready.")

In [ ]:
# ============================================================
# FILE DISCOVERY HELPERS
# ============================================================

METRIC_FILE_STEMS = {
    "global": "secondary_global_metrics",
    "regime": "secondary_regime_metrics",
    "tail": "secondary_tail_metrics",
    "braking_safety": "secondary_braking_safety_metrics",
    "severe_braking": "secondary_severe_braking_metrics",
    "sign": "secondary_sign_consistency_metrics",
    "critical": "secondary_critical_situation_metrics",
    "safety_summary": "secondary_final_safety_summary",
    "regime_mae_pivot": "secondary_regime_mae_pivot",
}


def horizon_tag(h):
    return f"H{HISTORY}_Hz{h}_S{STRIDE}"


def horizon_label(h):
    return f"{h / FPS:.2f}s"


def output_tag(h):
    return f"{horizon_tag(h)}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"


def newest(paths):
    paths = [Path(p) for p in paths if p is not None and Path(p).exists()]
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime)


def find_metric_file(metric_kind, h):
    stem = METRIC_FILE_STEMS[metric_kind]
    tag = output_tag(h)

    exact = SECONDARY_DIR / f"{stem}_{tag}.csv"
    if exact.exists():
        return exact

    # Flexible fallback in case an older output has a slightly different suffix.
    candidates = list(SECONDARY_DIR.glob(f"{stem}_*{horizon_tag(h)}*.csv"))
    candidates = [p for p in candidates if THRESHOLD_TAG in p.name and IDM_PARAM_TAG in p.name]

    return newest(candidates)


def find_config_file(h):
    tag = output_tag(h)
    exact = SECONDARY_DIR / f"secondary_config_{tag}.json"
    if exact.exists():
        return exact

    candidates = list(SECONDARY_DIR.glob(f"secondary_config_*{horizon_tag(h)}*.json"))
    candidates = [p for p in candidates if THRESHOLD_TAG in p.name and IDM_PARAM_TAG in p.name]

    return newest(candidates)

print("Discovery helpers ready.")

In [ ]:
# ============================================================
# BUILD FILE MANIFEST
# ============================================================

manifest_rows = []

for h in HORIZONS:
    seq_tag = horizon_tag(h)
    h_sec = h / FPS

    for metric_kind in METRIC_FILE_STEMS:
        p = find_metric_file(metric_kind, h)

        manifest_rows.append({
            "HorizonFrames": h,
            "HorizonSeconds": h_sec,
            "HorizonLabel": horizon_label(h),
            "SequenceTag": seq_tag,
            "MetricKind": metric_kind,
            "Found": p is not None,
            "FilePath": str(p) if p is not None else None,
        })

    cfg = find_config_file(h)
    manifest_rows.append({
        "HorizonFrames": h,
        "HorizonSeconds": h_sec,
        "HorizonLabel": horizon_label(h),
        "SequenceTag": seq_tag,
        "MetricKind": "config",
        "Found": cfg is not None,
        "FilePath": str(cfg) if cfg is not None else None,
    })

file_manifest = pd.DataFrame(manifest_rows)
file_manifest = file_manifest.sort_values(["HorizonFrames", "MetricKind"]).reset_index(drop=True)

availability = file_manifest.pivot_table(
    index="MetricKind",
    columns="HorizonLabel",
    values="Found",
    aggfunc="max",
    fill_value=False,
)

# Keep horizon columns in chronological order.
availability = availability[[horizon_label(h) for h in HORIZONS]]

file_manifest.to_csv(SUMMARY_DIR / "secondary_all_horizons_file_manifest.csv", index=False)
availability.to_csv(SUMMARY_DIR / "secondary_all_horizons_availability_matrix.csv")

print("=== Availability matrix ===")
display(availability)

print("\nManifest saved to:")
print(SUMMARY_DIR / "secondary_all_horizons_file_manifest.csv")

In [ ]:
# ============================================================
# LOAD ALL SECONDARY METRIC TABLES
# ============================================================

def load_metric_kind(metric_kind):
    frames = []

    for h in HORIZONS:
        p = find_metric_file(metric_kind, h)
        if p is None:
            print(f"Missing {metric_kind} for {horizon_tag(h)}")
            continue

        df = pd.read_csv(p)
        df["HorizonFrames"] = h
        df["HorizonSeconds"] = h / FPS
        df["HorizonLabel"] = horizon_label(h)
        df["SequenceTag"] = horizon_tag(h)
        df["SourceFile"] = str(p)

        if "Model" in df.columns:
            df = add_model_info(df)

        frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


tables = {kind: load_metric_kind(kind) for kind in METRIC_FILE_STEMS}

global_all = tables["global"]
regime_all = tables["regime"]
tail_all = tables["tail"]
braking_all = tables["braking_safety"]
severe_braking_all = tables["severe_braking"]
sign_all = tables["sign"]
critical_all = tables["critical"]
safety_all = tables["safety_summary"]
regime_pivot_all = tables["regime_mae_pivot"]

for name, df in tables.items():
    print(f"{name:20s}: {df.shape}")

# Save combined raw tables.
for name, df in tables.items():
    if len(df) > 0:
        df.to_csv(SUMMARY_DIR / f"secondary_all_horizons_{name}.csv", index=False)

print("\nCombined tables saved to:")
print(SUMMARY_DIR)

## Global metric summary tables

These tables compare the same model across horizons. The deployable table excludes true-regime and ideal upper-bound references.

In [ ]:
# ============================================================
# GLOBAL SUMMARY TABLES
# ============================================================

if len(global_all) == 0:
    raise RuntimeError("No global metrics were loaded. Run notebook 07 for at least one horizon first.")

metric_cols = [c for c in ["MAE", "RMSE", "MedAE", "R2", "Bias", "Corr", "Count"] if c in global_all.columns]

global_compact = global_all[
    ["HorizonFrames", "HorizonSeconds", "HorizonLabel", "Model", "ShortName", "Family", "Type"] + metric_cols
].sort_values(["HorizonFrames", "MAE"]).reset_index(drop=True)

deployable_global = global_compact[global_compact["Type"].eq("Deployable")].copy()
upper_bound_global = global_compact[global_compact["Type"].eq("Upper-bound")].copy()

best_deployable_by_horizon = (
    deployable_global
    .sort_values(["HorizonFrames", "MAE"])
    .groupby("HorizonFrames", as_index=False)
    .first()
)

best_upper_bound_by_horizon = (
    upper_bound_global
    .sort_values(["HorizonFrames", "MAE"])
    .groupby("HorizonFrames", as_index=False)
    .first()
)

mae_wide = global_compact.pivot_table(
    index=["Model", "ShortName", "Family", "Type"],
    columns="HorizonLabel",
    values="MAE",
    aggfunc="first",
).reset_index()

mae_wide = mae_wide[["Model", "ShortName", "Family", "Type"] + [horizon_label(h) for h in HORIZONS if horizon_label(h) in mae_wide.columns]]

global_compact.to_csv(SUMMARY_DIR / "table_global_metrics_all_horizons_long.csv", index=False)
deployable_global.to_csv(SUMMARY_DIR / "table_deployable_global_metrics_all_horizons.csv", index=False)
upper_bound_global.to_csv(SUMMARY_DIR / "table_upper_bound_global_metrics_all_horizons.csv", index=False)
best_deployable_by_horizon.to_csv(SUMMARY_DIR / "table_best_deployable_by_horizon.csv", index=False)
best_upper_bound_by_horizon.to_csv(SUMMARY_DIR / "table_best_upper_bound_by_horizon.csv", index=False)
mae_wide.to_csv(SUMMARY_DIR / "table_mae_wide_by_model_and_horizon.csv", index=False)

print("=== Best deployable model by horizon, using MAE ===")
display(best_deployable_by_horizon[["HorizonLabel", "Model", "MAE", "RMSE", "R2", "Bias", "Corr"]])

print("\n=== Best upper-bound model by horizon, using MAE ===")
display(best_upper_bound_by_horizon[["HorizonLabel", "Model", "MAE", "RMSE", "R2", "Bias", "Corr"]])

print("\n=== MAE wide table ===")
display(mae_wide)

## Regime and safety summary tables

These summarize braking/acceleration/steady performance and secondary safety metrics across horizons.

In [ ]:
# ============================================================
# REGIME AND SAFETY TABLES
# ============================================================

if len(regime_all) > 0:
    regime_mae_wide = regime_all.pivot_table(
        index=["Model", "ShortName", "Family", "Type", "Regime"],
        columns="HorizonLabel",
        values="MAE",
        aggfunc="first",
    ).reset_index()

    regime_mae_wide = regime_mae_wide[
        ["Model", "ShortName", "Family", "Type", "Regime"]
        + [horizon_label(h) for h in HORIZONS if horizon_label(h) in regime_mae_wide.columns]
    ]

    regime_mae_wide.to_csv(SUMMARY_DIR / "table_regime_mae_wide_by_model_and_horizon.csv", index=False)

    print("=== Regime MAE wide table ===")
    display(regime_mae_wide)
else:
    print("No regime metrics loaded.")


if len(safety_all) > 0:
    safety_cols = [
        "HorizonFrames", "HorizonSeconds", "HorizonLabel", "Model", "ShortName", "Family", "Type",
        "MAE", "RMSE", "P95_abs_error", "P99_abs_error", "LargeErrorRate_abs_gt_0p5",
        "Braking_MAE", "UnderBrake_Rate_gt_0p5", "WrongSign_Rate_in_Braking",
        "Predicts_SteadyOrAccel_Rate", "SignAccuracy", "BrakeRecall", "AccelRecall", "SteadyRecall",
        "SafetyRankScore",
    ]
    safety_cols = [c for c in safety_cols if c in safety_all.columns]

    safety_compact = safety_all[safety_cols].sort_values(["HorizonFrames", "SafetyRankScore" if "SafetyRankScore" in safety_all.columns else "MAE"])

    safety_compact.to_csv(SUMMARY_DIR / "table_safety_summary_all_horizons.csv", index=False)

    best_safety_by_horizon = (
        safety_compact
        .sort_values(["HorizonFrames", "SafetyRankScore" if "SafetyRankScore" in safety_compact.columns else "MAE"])
        .groupby("HorizonFrames", as_index=False)
        .first()
    )

    best_safety_by_horizon.to_csv(SUMMARY_DIR / "table_best_safety_rank_by_horizon.csv", index=False)

    print("\n=== Best safety-rank model by horizon ===")
    display(best_safety_by_horizon[[c for c in ["HorizonLabel", "Model", "SafetyRankScore", "MAE", "P95_abs_error", "UnderBrake_Rate_gt_0p5", "BrakeRecall"] if c in best_safety_by_horizon.columns]])
else:
    print("No safety summary metrics loaded.")

## Plot helpers

The plots below use multi-panel figures with a single shared legend outside the axes. Edit `PLOT_MODELS` to reduce clutter or change which models are shown.


In [ ]:
# ============================================================
# PLOT HELPERS — CLEAN MULTI-PANEL FIGURES WITH ONE SHARED LEGEND
# ============================================================

PLOT_MODELS = [m for m in DEFAULT_MODELS_FOR_PLOTS if m in set(global_all["Model"])]

# Recommended reduced set for clean thesis figures.
# Uncomment and edit if the legend is too crowded.
# PLOT_MODELS = [
#     "Plain Transformer",
#     "Global IDM",
#     "XGB-soft best-IDM selector",
#     "Residual Hybrid + Global IDM",
#     "Transformer Weighted Fusion + Global IDM",
#     "Transformer + XGB-soft best-IDM feature",
# ]

print("Models used in line plots:")
for m in PLOT_MODELS:
    print("-", MODEL_INFO.get(m, {}).get("ShortName", m))


# ============================================================
# GLOBAL PLOT SIZE SETTINGS
# Change these if the plots still look too wide/tall.
# ============================================================

FIGSIZE_2PANEL = (11.0, 4.3)      # two plots side by side
FIGSIZE_3PANEL = (11.0, 7.2)      # three plots + legend in fourth slot
FIGSIZE_4PANEL = (11.2, 7.4)      # four plots, legend below

LEGEND_NCOL_BOTTOM = 4
LEGEND_FONTSIZE = 9
LINE_WIDTH = 1.8
MARKER_SIZE = 5


def clean_filename(s):
    bad = ['/', '\\', ':', '*', '?', '"', '<', '>', '|', '+']
    out = str(s)
    for b in bad:
        out = out.replace(b, '_')
    out = out.replace(' ', '_')
    return out


def short_label(model_name):
    return MODEL_INFO.get(model_name, {}).get("ShortName", model_name)


def _unique_legend(handles, labels):
    unique = {}
    for h, lab in zip(handles, labels):
        if lab not in unique:
            unique[lab] = h
    return list(unique.values()), list(unique.keys())


def _filter_plot_df(df, metric, models=None, type_filter=None, family_filter=None):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    if metric not in df.columns:
        print(f"Metric {metric!r} not found.")
        return pd.DataFrame()

    plot_df = df.copy()

    if models is not None:
        plot_df = plot_df[plot_df["Model"].isin(models)].copy()

    if type_filter is not None and "Type" in plot_df.columns:
        plot_df = plot_df[plot_df["Type"].eq(type_filter)].copy()

    if family_filter is not None and "Family" in plot_df.columns:
        plot_df = plot_df[plot_df["Family"].eq(family_filter)].copy()

    plot_df = plot_df.dropna(subset=[metric, "HorizonSeconds", "Model"])

    return plot_df


def _plot_lines_on_axis(ax, df, metric, models=None, ylabel=None):
    plot_df = _filter_plot_df(df, metric, models=models)

    if len(plot_df) == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return [], []

    if models is None:
        model_order = list(plot_df["Model"].dropna().unique())
    else:
        model_order = [m for m in models if m in set(plot_df["Model"])]

    handles = []
    labels = []

    for model_name in model_order:
        g = plot_df[plot_df["Model"].eq(model_name)].sort_values("HorizonSeconds")

        if len(g) == 0:
            continue

        line, = ax.plot(
            g["HorizonSeconds"],
            g[metric],
            marker="o",
            linewidth=LINE_WIDTH,
            markersize=MARKER_SIZE,
            label=short_label(model_name),
        )
        handles.append(line)
        labels.append(short_label(model_name))

    ax.set_xlabel("Prediction horizon (s)")
    ax.set_ylabel(ylabel or metric)
    ax.grid(True, alpha=0.3)

    return handles, labels


def _apply_shared_legend(
    fig,
    axes_flat,
    nplots,
    handles,
    labels,
    legend_position="auto",
    legend_ncol=LEGEND_NCOL_BOTTOM,
):
    """
    Legend placement rules:
    - auto: 3 panels -> legend in empty subplot; 2 or 4 panels -> bottom
    - empty_axis: uses the first unused subplot as the legend area
    - bottom: places one legend below all panels
    - right: places one legend to the right
    """

    handles, labels = _unique_legend(handles, labels)

    if len(handles) == 0:
        fig.tight_layout()
        return

    total_axes = len(axes_flat)
    has_empty_axis = nplots < total_axes

    if legend_position == "auto":
        if has_empty_axis:
            legend_position = "empty_axis"
        else:
            legend_position = "bottom"

    if legend_position == "empty_axis" and has_empty_axis:
        legend_ax = axes_flat[nplots]
        legend_ax.axis("off")
        legend_ax.legend(
            handles,
            labels,
            loc="center",
            frameon=True,
            fontsize=LEGEND_FONTSIZE,
        )

        for j in range(nplots + 1, total_axes):
            axes_flat[j].axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return

    # Hide unused axes when the legend is not placed inside them.
    for j in range(nplots, total_axes):
        axes_flat[j].axis("off")

    if legend_position == "right":
        fig.legend(
            handles,
            labels,
            loc="center left",
            bbox_to_anchor=(0.985, 0.5),
            frameon=True,
            fontsize=LEGEND_FONTSIZE,
        )
        fig.tight_layout(rect=[0, 0, 0.82, 0.94])

    elif legend_position == "bottom":
        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.01),
            ncol=legend_ncol,
            frameon=True,
            fontsize=LEGEND_FONTSIZE,
        )
        fig.tight_layout(rect=[0, 0.12, 1, 0.94])

    else:
        fig.tight_layout(rect=[0, 0, 1, 0.94])


def _default_figsize(nplots, ncols):
    if nplots == 2:
        return FIGSIZE_2PANEL
    if nplots == 3:
        return FIGSIZE_3PANEL
    if nplots == 4:
        return FIGSIZE_4PANEL

    nrows = int(np.ceil(nplots / ncols))
    return (5.6 * ncols, 3.7 * nrows)


def plot_metric_grid(
    df,
    metrics,
    models=None,
    ncols=2,
    title=None,
    output_name=None,
    figsize=None,
    legend_position="auto",
    legend_ncol=LEGEND_NCOL_BOTTOM,
):
    """
    Plot 2, 3, or 4 metrics in one image with one shared legend.

    Recommended use:
    - 2 panels -> legend_position="bottom"
    - 3 panels -> legend_position="empty_axis"
    - 4 panels -> legend_position="bottom"
    - auto chooses those rules automatically.
    """

    valid_metrics = []
    for metric_col, subplot_title, ylabel in metrics:
        if df is not None and len(df) > 0 and metric_col in df.columns:
            valid_metrics.append((metric_col, subplot_title, ylabel))
        else:
            print(f"Skipping missing metric: {metric_col}")

    if len(valid_metrics) == 0:
        print("No valid metrics to plot.")
        return None

    nplots = len(valid_metrics)
    nrows = int(np.ceil(nplots / ncols))

    if figsize is None:
        figsize = _default_figsize(nplots, ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figsize,
        squeeze=False,
    )

    axes_flat = axes.flatten()
    all_handles = []
    all_labels = []

    for ax, (metric_col, subplot_title, ylabel) in zip(axes_flat, valid_metrics):
        handles, labels = _plot_lines_on_axis(ax, df, metric_col, models=models, ylabel=ylabel)
        ax.set_title(subplot_title)
        all_handles.extend(handles)
        all_labels.extend(labels)

    if title:
        fig.suptitle(title, y=0.99, fontsize=14)

    _apply_shared_legend(
        fig,
        axes_flat,
        nplots,
        all_handles,
        all_labels,
        legend_position=legend_position,
        legend_ncol=legend_ncol,
    )

    if output_name is not None:
        fig.savefig(SUMMARY_DIR / output_name, dpi=300, bbox_inches="tight")
        print("Saved:", SUMMARY_DIR / output_name)

    plt.show()
    return fig


def plot_subset_metric_grid(
    df,
    subset_col,
    subset_values,
    metric,
    ylabel,
    title_prefix,
    models=None,
    ncols=2,
    output_name=None,
    figsize=None,
    legend_position="auto",
    legend_ncol=LEGEND_NCOL_BOTTOM,
):
    """
    Plot the same metric for multiple subsets, e.g. regime MAE or critical-subset MAE.
    Each subset gets one panel. The legend is shared once.
    """

    if df is None or len(df) == 0:
        print("No data to plot.")
        return None

    if subset_col not in df.columns:
        print(f"Subset column {subset_col!r} not found.")
        return None

    if metric not in df.columns:
        print(f"Metric {metric!r} not found.")
        return None

    available = set(df[subset_col].dropna().astype(str))
    subset_values = [s for s in subset_values if str(s) in available]

    if len(subset_values) == 0:
        print(f"No requested {subset_col} values found.")
        print("Available:", sorted(available))
        return None

    nplots = len(subset_values)
    nrows = int(np.ceil(nplots / ncols))

    if figsize is None:
        figsize = _default_figsize(nplots, ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figsize,
        squeeze=False,
    )

    axes_flat = axes.flatten()
    all_handles = []
    all_labels = []

    for ax, subset_value in zip(axes_flat, subset_values):
        subset_df = df[df[subset_col].astype(str).eq(str(subset_value))].copy()
        handles, labels = _plot_lines_on_axis(ax, subset_df, metric, models=models, ylabel=ylabel)
        ax.set_title(f"{title_prefix}: {subset_value}")
        all_handles.extend(handles)
        all_labels.extend(labels)

    _apply_shared_legend(
        fig,
        axes_flat,
        nplots,
        all_handles,
        all_labels,
        legend_position=legend_position,
        legend_ncol=legend_ncol,
    )

    if output_name is not None:
        fig.savefig(SUMMARY_DIR / output_name, dpi=300, bbox_inches="tight")
        print("Saved:", SUMMARY_DIR / output_name)

    plt.show()
    return fig


def plot_ranked_bar_at_horizon(df, metric, horizon, title=None, output_name=None, ascending=True, top_n=None):
    if df is None or len(df) == 0:
        print("No data.")
        return None

    if metric not in df.columns:
        print(f"Metric {metric!r} not found.")
        return None

    plot_df = df[df["HorizonFrames"].eq(horizon)].copy()

    if len(plot_df) == 0:
        print(f"No rows found for horizon {horizon}.")
        return None

    plot_df = plot_df.sort_values(metric, ascending=ascending)

    if top_n is not None:
        plot_df = plot_df.head(top_n)

    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(plot_df))))
    y = np.arange(len(plot_df))

    labels = plot_df["ShortName"] if "ShortName" in plot_df.columns else plot_df["Model"]

    ax.barh(y, plot_df[metric])
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_title(title or f"{metric} at {horizon / FPS:.2f}s")
    ax.set_xlabel(metric)
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()

    if output_name is not None:
        fig.savefig(SUMMARY_DIR / output_name, dpi=300, bbox_inches="tight")
        print("Saved:", SUMMARY_DIR / output_name)

    plt.show()
    return fig


print("Clean-layout multi-panel plot helpers ready.")


## Global performance multi-panel plots

Each image contains two or four subplots with one shared legend outside the plotting area.


In [ ]:
# ============================================================
# GLOBAL PERFORMANCE MULTI-PANEL PLOTS
# ============================================================

# Main 4-panel figure for thesis/results overview.
global_metrics_4panel = [
    ("MAE", "Global MAE", "MAE (m/s²)"),
    ("RMSE", "Global RMSE", "RMSE (m/s²)"),
    ("R2", "Global R²", "R²"),
    ("Corr", "Correlation", "Correlation"),
]

plot_metric_grid(
    global_all,
    metrics=global_metrics_4panel,
    models=PLOT_MODELS,
    ncols=2,
    title="Global performance across prediction horizons",
    output_name="multipanel_global_performance_4panel.png",
    figsize=FIGSIZE_4PANEL,
    legend_position="bottom",
    legend_ncol=LEGEND_NCOL_BOTTOM,
)

# Optional 2-panel version for a cleaner accuracy-only figure.
global_accuracy_2panel = [
    ("MAE", "Global MAE", "MAE (m/s²)"),
    ("RMSE", "Global RMSE", "RMSE (m/s²)"),
]

plot_metric_grid(
    global_all,
    metrics=global_accuracy_2panel,
    models=PLOT_MODELS,
    ncols=2,
    title="Global accuracy across prediction horizons",
    output_name="multipanel_global_accuracy_2panel.png",
    figsize=FIGSIZE_2PANEL,
    legend_position="bottom",
    legend_ncol=LEGEND_NCOL_BOTTOM,
)


## Safety-oriented multi-panel plots

These figures summarize tail error, under-braking, brake recall, and sign/regime consistency across horizons.


In [ ]:
# ============================================================
# SAFETY-ORIENTED MULTI-PANEL PLOTS
# ============================================================

if len(safety_all) > 0:
    safety_metrics_4panel = [
        ("P95_abs_error", "P95 absolute error", "P95 abs. error (m/s²)"),
        ("UnderBrake_Rate_gt_0p5", "Strong under-braking rate", "Rate"),
        ("BrakeRecall", "Brake recall", "Recall"),
        ("SignAccuracy", "Sign/regime accuracy", "Accuracy"),
    ]

    plot_metric_grid(
        safety_all,
        metrics=safety_metrics_4panel,
        models=PLOT_MODELS,
        ncols=2,
        title="Safety-oriented metrics across prediction horizons",
        output_name="multipanel_safety_metrics_4panel.png",
        figsize=FIGSIZE_4PANEL,
        legend_position="bottom",
        legend_ncol=LEGEND_NCOL_BOTTOM,
    )

    braking_safety_2panel = [
        ("UnderBrake_Rate_gt_0p5", "Strong under-braking rate", "Rate"),
        ("WrongSign_Rate_in_Braking", "Wrong-sign rate in braking", "Rate"),
    ]

    plot_metric_grid(
        safety_all,
        metrics=braking_safety_2panel,
        models=PLOT_MODELS,
        ncols=2,
        title="Braking safety across prediction horizons",
        output_name="multipanel_braking_safety_2panel.png",
        figsize=FIGSIZE_2PANEL,
        legend_position="bottom",
        legend_ncol=LEGEND_NCOL_BOTTOM,
    )
else:
    print("No safety summary table loaded.")


## Regime-specific multi-panel plots

The regime figures compare acceleration, braking, and steady-driving errors in a shared layout.


In [ ]:
# ============================================================
# REGIME-SPECIFIC MULTI-PANEL PLOTS
# ============================================================

if len(regime_all) > 0:
    regime_values = ["accelerating", "braking", "steady"]

    # 3-panel figure: legend uses the empty fourth subplot.
    plot_subset_metric_grid(
        regime_all.assign(Regime=regime_all["Regime"].astype(str).str.lower()),
        subset_col="Regime",
        subset_values=regime_values,
        metric="MAE",
        ylabel="MAE (m/s²)",
        title_prefix="Regime MAE",
        models=PLOT_MODELS,
        ncols=2,
        output_name="multipanel_regime_mae_3panel.png",
        figsize=FIGSIZE_3PANEL,
        legend_position="empty_axis",
    )

    # Optional 2-panel figure focused on dynamic regimes only.
    plot_subset_metric_grid(
        regime_all.assign(Regime=regime_all["Regime"].astype(str).str.lower()),
        subset_col="Regime",
        subset_values=["accelerating", "braking"],
        metric="MAE",
        ylabel="MAE (m/s²)",
        title_prefix="Dynamic-regime MAE",
        models=PLOT_MODELS,
        ncols=2,
        output_name="multipanel_dynamic_regime_mae_2panel.png",
        figsize=FIGSIZE_2PANEL,
        legend_position="bottom",
        legend_ncol=LEGEND_NCOL_BOTTOM,
    )
else:
    print("No regime table loaded.")


## Critical-situation multi-panel plots

The critical-situation figures place several critical subsets in one image with one shared legend outside the axes.


In [ ]:
# ============================================================
# CRITICAL-SITUATION MULTI-PANEL PLOTS
# ============================================================

if len(critical_all) > 0:
    available_subsets = list(critical_all["Subset"].dropna().astype(str).unique())

    # Put the most interpretable safety subsets first if they exist.
    preferred_subsets = [
        "CloseFollowing",
        "HighInvTTC",
        "LowHeadway",
        "Braking_NearLaneChange",
        "NearLaneChange",
        "CriticalClosing",
    ]

    critical_subsets_to_plot = [s for s in preferred_subsets if s in available_subsets]
    critical_subsets_to_plot += [s for s in available_subsets if s not in critical_subsets_to_plot]
    critical_subsets_to_plot = critical_subsets_to_plot[:4]

    print("Critical subsets plotted:")
    for s in critical_subsets_to_plot:
        print("-", s)

    # 4-panel figure: critical-subset MAE.
    plot_subset_metric_grid(
        critical_all,
        subset_col="Subset",
        subset_values=critical_subsets_to_plot,
        metric="MAE",
        ylabel="MAE (m/s²)",
        title_prefix="Critical-subset MAE",
        models=PLOT_MODELS,
        ncols=2,
        output_name="multipanel_critical_subset_mae_4panel.png",
        figsize=FIGSIZE_4PANEL,
        legend_position="bottom",
        legend_ncol=LEGEND_NCOL_BOTTOM,
    )

    # 4-panel figure: critical-subset under-braking rate, if available.
    if "UnderBrake_Rate_gt_0p5" in critical_all.columns:
        plot_subset_metric_grid(
            critical_all,
            subset_col="Subset",
            subset_values=critical_subsets_to_plot,
            metric="UnderBrake_Rate_gt_0p5",
            ylabel="Rate",
            title_prefix="Critical-subset under-braking",
            models=PLOT_MODELS,
            ncols=2,
            output_name="multipanel_critical_subset_underbrake_4panel.png",
            figsize=FIGSIZE_4PANEL,
            legend_position="bottom",
            legend_ncol=LEGEND_NCOL_BOTTOM,
        )
else:
    print("No critical-situation table loaded.")


## Ranked model comparisons at a chosen horizon

Change `RANK_HORIZON` to inspect another horizon.

In [ ]:
# ============================================================
# RANKED MODEL BARS AT ONE HORIZON
# ============================================================

RANK_HORIZON = 125

plot_ranked_bar_at_horizon(
    global_all,
    metric="MAE",
    horizon=RANK_HORIZON,
    title=f"Global MAE ranking at {RANK_HORIZON / FPS:.2f}s",
    output_name=f"plot_ranked_global_mae_Hz{RANK_HORIZON}.png",
    ascending=True,
)

if len(safety_all) > 0 and "UnderBrake_Rate_gt_0p5" in safety_all.columns:
    plot_ranked_bar_at_horizon(
        safety_all,
        metric="UnderBrake_Rate_gt_0p5",
        horizon=RANK_HORIZON,
        title=f"Strong under-braking ranking at {RANK_HORIZON / FPS:.2f}s",
        output_name=f"plot_ranked_underbrake_Hz{RANK_HORIZON}.png",
        ascending=True,
    )

## Thesis-ready summary tables

These cells make compact tables that are easier to paste into the thesis or export to CSV.

In [ ]:
# ============================================================
# THESIS-READY TABLES
# ============================================================

# 1) Compact deployable table: MAE/RMSE/R2 across horizons.
deployable_core = deployable_global.pivot_table(
    index=["Model", "ShortName", "Family"],
    columns="HorizonLabel",
    values="MAE",
    aggfunc="first",
).reset_index()

deployable_core = deployable_core[["Model", "ShortName", "Family"] + [horizon_label(h) for h in HORIZONS if horizon_label(h) in deployable_core.columns]]

deployable_core.to_csv(SUMMARY_DIR / "thesis_table_deployable_mae_by_horizon.csv", index=False)

print("=== Thesis table: deployable MAE by horizon ===")
display(deployable_core)


# 2) Compact safety table: under-braking rate across horizons.
if len(safety_all) > 0 and "UnderBrake_Rate_gt_0p5" in safety_all.columns:
    underbrake_wide = safety_all.pivot_table(
        index=["Model", "ShortName", "Family", "Type"],
        columns="HorizonLabel",
        values="UnderBrake_Rate_gt_0p5",
        aggfunc="first",
    ).reset_index()

    underbrake_wide = underbrake_wide[["Model", "ShortName", "Family", "Type"] + [horizon_label(h) for h in HORIZONS if horizon_label(h) in underbrake_wide.columns]]
    underbrake_wide.to_csv(SUMMARY_DIR / "thesis_table_underbraking_rate_by_horizon.csv", index=False)

    print("\n=== Thesis table: under-braking rate by horizon ===")
    display(underbrake_wide)


# 3) Best models by horizon, deployable only.
best_deployable_simple = best_deployable_by_horizon[[
    c for c in ["HorizonLabel", "Model", "MAE", "RMSE", "R2", "Bias", "Corr"]
    if c in best_deployable_by_horizon.columns
]].copy()

best_deployable_simple.to_csv(SUMMARY_DIR / "thesis_table_best_deployable_by_horizon.csv", index=False)

print("\n=== Thesis table: best deployable model by horizon ===")
display(best_deployable_simple)

## File keep recommendations for secondary outputs

This does not delete anything. It only creates a CSV recommendation list.

In [ ]:
# ============================================================
# SECONDARY OUTPUT FILE KEEP RECOMMENDATIONS
# ============================================================

KEEP_KINDS = {
    "global",
    "regime",
    "tail",
    "braking_safety",
    "sign",
    "critical",
    "safety_summary",
    "config",
}

OPTIONAL_KINDS = {
    "severe_braking",
    "regime_mae_pivot",
}

keep_manifest = file_manifest.copy()

keep_manifest["Recommendation"] = np.where(
    keep_manifest["Found"].eq(False),
    "MISSING",
    np.where(
        keep_manifest["MetricKind"].isin(KEEP_KINDS),
        "KEEP",
        np.where(
            keep_manifest["MetricKind"].isin(OPTIONAL_KINDS),
            "ARCHIVE_OPTIONAL",
            "CHECK",
        ),
    ),
)

keep_manifest.to_csv(SUMMARY_DIR / "secondary_all_horizons_keep_recommendations.csv", index=False)

print("=== Keep recommendation summary ===")
display(keep_manifest.groupby(["MetricKind", "Recommendation"]).size().reset_index(name="Count"))

print("\nDetailed keep manifest saved to:")
print(SUMMARY_DIR / "secondary_all_horizons_keep_recommendations.csv")

In [ ]:
# ============================================================
# FINAL OUTPUT LIST
# ============================================================

print("Summary outputs written to:")
print(SUMMARY_DIR)

print("\nCSV outputs:")
for p in sorted(SUMMARY_DIR.glob("*.csv")):
    print("-", p.name)

print("\nPlot outputs:")
for p in sorted(SUMMARY_DIR.glob("*.png")):
    print("-", p.name)